  # Sample design: a reproducible Gaia DR3 sample
  Aim: pull a fixed, reproducible, random set of stars (same query always returns the same stars), with enough stars per class for the later tests.

   ## RR Lyrae sample (reproducible)
   Gaia DR3, `gaiadr3.vari_rrlyrae` joined to `gaia_source`. Stars with random_index < 20,000,000, ordered by random_index, first 2000. The same query returns the same stars (checked on the full table). Query date 2026-10-13.

In [1]:
from astroquery.gaia import Gaia
import pandas as pd
from datetime import date

query = """
SELECT TOP 2000
    r.source_id, r.pf, r.p1_o, r.peak_to_peak_g, r.num_clean_epochs_g,
    g.random_index, g.phot_g_mean_mag, g.bp_rp, g.parallax_over_error
FROM gaiadr3.vari_rrlyrae AS r
JOIN gaiadr3.gaia_source AS g USING (source_id)
WHERE g.random_index < 20000000
ORDER BY g.random_index
"""

rr_a = Gaia.launch_job(query).get_results().to_pandas()

print("Query date:", date.today(), "| Release: Gaia DR3")
print(rr_a.shape)
print(rr_a["source_id"].head(3).tolist())

In preparation for Gaia DR4, the Gaia archive is in evolution. Unfortunately, it may be unstable at times and particular types of queries may time out. Please consider registering for a user account (https://www.cosmos.esa.int/web/gaia-users/register). For questions or advice, please contact the Gaia helpdesk (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk).
Query date: 2026-10-13 | Release: Gaia DR3
(2000, 9)
[5926547345107100416, 4229457091843346688, 6059494724859516800]


In [2]:
rr_a.to_csv("../data/raw/gaia_dr3_rrlyrae_sample_ri20M_raw.csv", index=False)
print("saved", rr_a.shape)

saved (2000, 9)


In [3]:
saved = pd.read_csv("../data/raw/gaia_dr3_rrlyrae_sample_ri20M_raw.csv")
again = Gaia.launch_job(query).get_results().to_pandas()

same_ids = saved["source_id"].equals(again["source_id"])
print("all 2000 source_ids identical and in the same order:", same_ids)

all 2000 source_ids identical and in the same order: True


In [4]:
def pull_class(class_name, n=1000, pool=20_000_000):
    q = f"""
    SELECT TOP {n}
        v.source_id, v.best_class_name, v.best_class_score,
        s.num_selected_g_fov, s.mean_mag_g_fov, s.range_mag_g_fov,
        s.trimmed_range_mag_g_fov, s.std_dev_mag_g_fov,
        g.random_index, g.phot_g_mean_mag, g.bp_rp, g.parallax_over_error
    FROM gaiadr3.vari_classifier_result AS v
    JOIN gaiadr3.vari_summary AS s USING (source_id)
    JOIN gaiadr3.gaia_source AS g USING (source_id)
    WHERE v.best_class_name = '{class_name}'
      AND g.random_index < {pool}
    ORDER BY g.random_index
    """
    return Gaia.launch_job(q).get_results().to_pandas()

ecl = pull_class("ECL")
print(ecl.shape)
print(ecl["source_id"].head(3).tolist())
print(ecl["best_class_score"].describe().round(2))

(1000, 12)
[5336426878835464960, 6026593969805570048, 211945744020716672]
count    1000.00
mean        0.54
std         0.29
min         0.00
25%         0.27
50%         0.56
75%         0.80
max         1.00
Name: best_class_score, dtype: float64


In [5]:
classes = ["ECL", "DSCT|GDOR|SXPHE", "SOLAR_LIKE", "RS", "S", "LPV", "RR", "CEP", "CV"]

parts = []
for c in classes:
    try:
        d = pull_class(c, n=1000, pool=20_000_000)
        print(f"{c:18s} {len(d):5d} stars")
        parts.append(d)
    except Exception as e:
        print(f"{c:18s} FAILED: {str(e)[:60]}")

classified = pd.concat(parts, ignore_index=True)
print()
print("Query date:", date.today(), "| Release: Gaia DR3 | pool: random_index < 20,000,000")
print(classified.shape)

classified.to_csv("../data/raw/gaia_dr3_classified_sample_ri20M_raw.csv", index=False)

ECL                 1000 stars
DSCT|GDOR|SXPHE     1000 stars
SOLAR_LIKE          1000 stars
RS                  1000 stars
S                   1000 stars
LPV                 1000 stars
RR                  1000 stars
CEP                  188 stars
CV                    69 stars

Query date: 2026-10-13 | Release: Gaia DR3 | pool: random_index < 20,000,000
(7257, 12)
